# Structural Estimation

This notebook implements the dynamic discrete choice model from `slides.tex`.

In [1]:
# Packages
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax.scipy.special import logsumexp as jax_logsumexp

### Load Data

In [2]:
df = pd.read_csv("./kkbox-subsample/panel_user_month.csv")

# Subsample 20 users
user_ids = pd.Series(df["msno"].unique()).iloc[:100].tolist()
df = df[df["msno"].isin(user_ids)].copy()

# Map observed statuses to model actions
status_to_action = {
    "churn": 0,
    "active": 1,
    "paused": 2,
}
df["action"] = df["subscription_status"].map(status_to_action)

df = df.sort_values(["msno", "user_tenure"]).reset_index(drop=True)

# Previous action for inertia term; initialize first month as previous active
df["prev_action"] = df.groupby("msno")["action"].shift(1)
df["prev_action"] = df["prev_action"].fillna(1).astype(int)

df["R_i"] = df.groupby("msno")["auto_renewal"].transform("first").astype(float)
df["p_i"] = df.groupby("msno")["price"].transform("first").astype(float)

df.head()

,msno,user_tenure,subscription_status,monthly_hours,cum_hours,auto_renewal,transaction_date,membership_expire_date,price,action,prev_action,R_i,p_i
0,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,0,active,39.826280,0.000000,0.0,20150215.0,20150317.0,149.0,1,1,0.0,149.0
1,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,1,active,16.865030,39.826280,0.0,20150320.0,20150419.0,149.0,1,1,0.0,149.0
2,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,2,active,20.236815,56.691311,0.0,20150421.0,20150521.0,149.0,1,1,0.0,149.0
3,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,3,active,17.657869,76.928126,0.0,20150522.0,20150621.0,149.0,1,1,0.0,149.0
4,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,4,active,11.805867,94.585995,0.0,20150621.0,20150721.0,149.0,1,1,0.0,149.0


In [3]:
# Discretize accumulated hours into bins

hour_edges = np.array([0.0, 10.0, 50.0, 100.0, 300.0, np.inf])
H = len(hour_edges) - 1

df["h_bin"] = pd.cut(
    df["cum_hours"],
    bins=hour_edges,
    include_lowest=True,
    labels=False,
).astype(int)

df[["msno", "cum_hours", "h_bin"]].head(10)

,msno,cum_hours,h_bin
0,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,0.000000,0
1,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,39.826280,1
2,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,56.691311,2
3,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,76.928126,2
4,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,94.585995,2
5,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,106.391862,3
6,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,108.420437,3
7,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,108.420437,3
8,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,121.918446,3
9,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,122.889533,3


In [4]:
# Build empirical active-user transition matrix for h_bin
# T[h, h'] = P(h_{t+1}=h' | h_t=h, a_t=1)

active_rows = df.copy()
active_rows["h_bin_next"] = active_rows.groupby("msno")["h_bin"].shift(-1)
active_rows = active_rows[(active_rows["action"] == 1) & (active_rows["h_bin_next"].notna())].copy()
active_rows["h_bin_next"] = active_rows["h_bin_next"].astype(int)
counts = np.zeros((H, H), dtype=float)
for _, r in active_rows.iterrows():
    counts[int(r["h_bin"]), int(r["h_bin_next"])] += 1.0
T = np.zeros((H, H), dtype=float)
for h in range(H):
    row_sum = counts[h].sum()
    if row_sum > 0:
        T[h] = counts[h] / row_sum
    else:
        T[h, h] = 1.0

print(T)

[[0.68103448 0.22413793 0.06896552 0.02586207 0.        ]
 [0.         0.66055046 0.33027523 0.00917431 0.        ]
 [0.         0.         0.64912281 0.35087719 0.        ]
 [0.         0.         0.         0.88940092 0.11059908]
 [0.         0.         0.         0.         1.        ]]


In [5]:
# Enumerate the dynamic state space
# Omega_t = (h_t, a_{t-1})

a_prev_vals = np.array([0, 1, 2], dtype=int)

state_tuples = []
for h in range(H):
    for a_prev in a_prev_vals:
        state_tuples.append((int(h), int(a_prev)))

state_to_idx = {s: i for i, s in enumerate(state_tuples)}
idx_to_state = {i: s for s, i in state_to_idx.items()}

n_states = len(state_tuples)
n_actions = 3  # churn, active, pause

In [6]:
# Flow utility specification from slides
# v0 = 0
# v1 = u1_bar + alpha*R - beta*p + eta*1{a_prev=1} + gamma_h[h]
# v2 = u2_bar + eta*1{a_prev=2}


def flow_utility_by_state(theta, state_tuples, H, R_i, p_i):
    """
    Returns U with shape (n_actions, n_states), where:
      U[a, s] = deterministic flow utility of action a in state s.
    """
    u1_bar = float(theta["u1_bar"])
    u2_bar = float(theta["u2_bar"])
    alpha = float(theta["alpha"])
    beta = float(theta["beta"])
    eta = float(theta["eta"])
    gamma = np.asarray(theta["gamma"], dtype=float).reshape(H)

    U = np.zeros((3, n_states), dtype=float)

    for s_idx, (h_bin, a_prev) in enumerate(state_tuples):
        # Action 0: churn
        U[0, s_idx] = 0.0

        # Action 1: active
        U[1, s_idx] = (
            u1_bar + alpha * float(R_i) - beta * float(p_i)
            + eta * (1 if a_prev == 1 else 0)
            + gamma[h_bin]
        )

        # Action 2: pause
        U[2, s_idx] = u2_bar + eta * (1 if a_prev == 2 else 0)

    return U

In [7]:
# Transition matrices P[a, s, s_next]
# Build action-specific transition tensor under model assumptions:
#   - a=0 churn: absorbing terminal action with zero continuation value,
#                implemented as self-transition (continuation still zero since v0=0 forever)
#   - a=1 active: h transitions via empirical T, next a_prev = 1
#   - a=2 pause: h is frozen, next a_prev = 2

P = np.zeros((3, n_states, n_states), dtype=float)

for s_idx, (h_bin, a_prev) in enumerate(state_tuples):
    # a = 0 (churn): absorbing for practical purposes
    P[0, s_idx, s_idx] = 1.0

    # a = 1 (active): draw h' from empirical active transition, set a_prev' = 1
    for h_next in range(H):
        s_next = (int(h_next), 1)
        s_next_idx = state_to_idx[s_next]
        P[1, s_idx, s_next_idx] += T[h_bin, h_next]

    # a = 2 (pause): keep h fixed, set a_prev' = 2
    s_next_pause = (int(h_bin), 2)
    s_next_pause_idx = state_to_idx[s_next_pause]
    P[2, s_idx, s_next_pause_idx] = 1.0

print("Transition tensor shape:", P.shape)

Transition tensor shape: (3, 15, 15)


In [8]:
# Bellman fixed-point solver (Type-I EV shocks)


def bellman_operator(Nu, U, P, delta):
    """
    One Bellman update on action-specific value function Nu.

    Nu shape: (n_actions, n_states)
    U  shape: (n_actions, n_states)
    P  shape: (n_actions, n_states, n_states)
    """
    cont_vals = logsumexp(Nu, axis=0)

    # Expected continuation value for each action-state pair
    EV = np.zeros_like(U)
    for a in range(U.shape[0]):
        EV[a] = P[a] @ cont_vals

    return U + delta * EV


def solve_fixed_point(U, P, delta=0.99, tol=1e-10, max_iter=10000):
    """
    Solve Nu = T(Nu) by value iteration.
    """
    Nu = np.zeros_like(U)

    for _ in range(max_iter):
        Nu_next = bellman_operator(Nu, U, P, delta)
        diff = np.max(np.abs(Nu_next - Nu))
        Nu = Nu_next
        if diff < tol:
            break

    return Nu


def choice_probabilities(Nu):
    """
    Return CCPs with shape (n_states, n_actions).
    """
    log_denom = logsumexp(Nu, axis=0, keepdims=True)
    log_pr = Nu - log_denom
    return np.exp(log_pr).T

In [9]:
# Helper: map observed panel rows to state indices used by DP

def attach_state_index(df_panel, state_to_idx):
    df_out = df_panel.copy()
    df_out["state_tuple"] = list(
        zip(
            df_out["h_bin"].astype(int),
            df_out["prev_action"].astype(int),
        )
    )
    df_out["state_idx"] = df_out["state_tuple"].map(state_to_idx)

    missing = df_out["state_idx"].isna().sum()
    if missing > 0:
        raise ValueError(f"Found {missing} rows with state not in state space.")

    df_out["state_idx"] = df_out["state_idx"].astype(int)
    return df_out


df_model = attach_state_index(df, state_to_idx)
df_model[["msno", "user_tenure", "action", "prev_action", "h_bin", "state_idx", "R_i", "p_i"]].head(10)

,msno,user_tenure,action,prev_action,h_bin,state_idx,R_i,p_i
0,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,0,1,1,0,1,0.0,149.0
1,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,1,1,1,1,4,0.0,149.0
2,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,2,1,1,2,7,0.0,149.0
3,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,3,1,1,2,7,0.0,149.0
4,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,4,1,1,2,7,0.0,149.0
5,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,5,1,1,3,10,0.0,149.0
6,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,6,2,1,3,10,0.0,149.0
7,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,7,1,2,3,11,0.0,149.0
8,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,8,1,1,3,10,0.0,149.0
9,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,9,0,1,3,10,0.0,149.0


In [10]:
# Fast path prep: vectorized arrays for type/state indexing

# Keep one integer index per (R_i, p_i) type
unique_types = (
    df_model[["R_i", "p_i"]]
    .drop_duplicates()
    .sort_values(["R_i", "p_i"])
    .reset_index(drop=True)
)

# Build map from tuple -> type index (robust; no tuple-attribute access)
type_pairs = list(zip(
    unique_types["R_i"].to_numpy(dtype=float),
    unique_types["p_i"].to_numpy(dtype=float),
))
type_to_idx = {pair: i for i, pair in enumerate(type_pairs)}

# Observation-level index arrays (used inside objective)
s_idx_np = df_model["state_idx"].to_numpy(dtype=int)
a_obs_np = df_model["action"].to_numpy(dtype=int)
type_idx_np = np.array([
    type_to_idx[(float(R_i), float(p_i))]
    for R_i, p_i in zip(df_model["R_i"].to_numpy(), df_model["p_i"].to_numpy())
], dtype=int)

# State-level arrays for vectorized utility construction
h_of_state_np = np.array([s[0] for s in state_tuples], dtype=int)
a_prev_of_state_np = np.array([s[1] for s in state_tuples], dtype=int)

# Type-level arrays
R_type_np = unique_types["R_i"].to_numpy(dtype=float)
p_type_np = unique_types["p_i"].to_numpy(dtype=float)

print("n_types:", len(unique_types), "n_states:", len(state_tuples), "n_obs:", len(df_model))

n_types: 8 n_states: 15 n_obs: 884


In [11]:
# Fast path: JAX objective + analytic gradient + vectorized Bellman/likelihood


def _unpack_theta_jax(theta_vec, H):
    return {
        "u1_bar": theta_vec[0],
        "u2_bar": theta_vec[1],
        "alpha": theta_vec[2],
        "beta": theta_vec[3],
        "eta": theta_vec[4],
        "gamma": theta_vec[5:5 + H],
    }


def _build_U_all_types_jax(theta, H, R_type, p_type, h_of_state, a_prev_of_state):
    """
    Returns U_all with shape (n_types, 3, n_states).
    """
    u1_bar = theta["u1_bar"]
    u2_bar = theta["u2_bar"]
    alpha = theta["alpha"]
    beta = theta["beta"]
    eta = theta["eta"]
    gamma = theta["gamma"]

    # Per-state terms
    gamma_state = gamma[h_of_state]                               # (S,)
    inertia_active = (a_prev_of_state == 1).astype(jnp.float64)  # (S,)
    inertia_pause = (a_prev_of_state == 2).astype(jnp.float64)   # (S,)

    # Type-specific level for active utility
    active_type_level = (u1_bar + alpha * R_type - beta * p_type)[:, None]  # (T,1)

    n_types = R_type.shape[0]
    n_states = h_of_state.shape[0]

    U0 = jnp.zeros((n_types, n_states))
    U1 = active_type_level + eta * inertia_active[None, :] + gamma_state[None, :]

    # Pause utility is type-invariant, so explicitly broadcast to (T,S)
    U2_base = u2_bar + eta * inertia_pause[None, :]
    U2 = jnp.broadcast_to(U2_base, (n_types, n_states))

    return jnp.stack([U0, U1, U2], axis=1)  # (T,3,S)


def _bellman_solve_all_types_jax(U_all, P_jax, delta, max_iter=500):
    """
    Batched value iteration over all types with fixed loop count.
    Using fixed iterations avoids reverse-mode autodiff issues with dynamic while loops.

    U_all: (T, A, S), P_jax: (A, S, S)
    Returns Nu_all: (T, A, S)
    """
    Nu0 = jnp.zeros_like(U_all)

    def body_fun(i, Nu):
        cont_vals = jax_logsumexp(Nu, axis=1)             # (T,S)
        EV = jnp.einsum("ask,tk->tas", P_jax, cont_vals) # (T,A,S)
        Nu_next = U_all + delta * EV
        return Nu_next

    Nu_star = jax.lax.fori_loop(0, max_iter, body_fun, Nu0)
    return Nu_star


def _neg_loglik_jax(theta_vec, arrays, H, delta):
    """
    Fully vectorized negative log-likelihood in JAX.
    """
    theta = _unpack_theta_jax(theta_vec, H)

    U_all = _build_U_all_types_jax(
        theta=theta,
        H=H,
        R_type=arrays["R_type"],
        p_type=arrays["p_type"],
        h_of_state=arrays["h_of_state"],
        a_prev_of_state=arrays["a_prev_of_state"],
    )

    Nu_all = _bellman_solve_all_types_jax(U_all, arrays["P"], delta=delta, max_iter=500)

    # CCP per type/state/action: shape (T,S,A)
    log_denom = jax_logsumexp(Nu_all, axis=1, keepdims=True)      # (T,1,S)
    CCP_all = jnp.exp((Nu_all - log_denom).transpose(0, 2, 1))    # (T,S,A)

    # Vectorized gather of observed probabilities
    p_obs = CCP_all[arrays["type_idx"], arrays["s_idx"], arrays["a_obs"]]
    p_obs = jnp.clip(p_obs, 1e-300, 1.0)

    return -jnp.sum(jnp.log(p_obs))


def estimate_model_fast(
    theta_init,
    H,
    delta,
    P,
    R_type,
    p_type,
    h_of_state,
    a_prev_of_state,
    type_idx,
    s_idx,
    a_obs,
    opt_method="L-BFGS-B",
):
    """
    Drop-in faster estimator:
      - JAX JIT objective
      - JAX analytic gradient
      - SciPy optimizer wrapper
    """
    arrays = {
        "P": jnp.asarray(P, dtype=jnp.float64),
        "R_type": jnp.asarray(R_type, dtype=jnp.float64),
        "p_type": jnp.asarray(p_type, dtype=jnp.float64),
        "h_of_state": jnp.asarray(h_of_state, dtype=jnp.int32),
        "a_prev_of_state": jnp.asarray(a_prev_of_state, dtype=jnp.int32),
        "type_idx": jnp.asarray(type_idx, dtype=jnp.int32),
        "s_idx": jnp.asarray(s_idx, dtype=jnp.int32),
        "a_obs": jnp.asarray(a_obs, dtype=jnp.int32),
    }

    obj = lambda th: _neg_loglik_jax(th, arrays=arrays, H=H, delta=delta)
    obj_jit = jax.jit(obj)
    grad_jit = jax.jit(jax.grad(obj))

    def obj_np(th):
        return float(obj_jit(jnp.asarray(th, dtype=jnp.float64)))

    def grad_np(th):
        return np.asarray(grad_jit(jnp.asarray(th, dtype=jnp.float64)), dtype=float)

    result = minimize(
        fun=obj_np,
        jac=grad_np,
        x0=np.asarray(theta_init, dtype=float),
        method=opt_method,
    )

    output = {
        "success": result.success,
        "message": result.message,
        "opt_nll": float(result.fun),
        "opt_theta_vec": result.x,
        "opt_theta": {
            "u1_bar": result.x[0],
            "u2_bar": result.x[1],
            "alpha": result.x[2],
            "beta": result.x[3],
            "eta": result.x[4],
            "gamma": result.x[5:5 + H],
        },
        "num_iter": result.nit,
    }
    return output

In [12]:
# Canonical fast-run cell (robust to out-of-order execution)
# This cell rebuilds all fast-path arrays safely, then runs estimate_model_fast.

# 1) Build unique (R_i, p_i) type table and mapping
unique_types = (
    df_model[["R_i", "p_i"]]
    .drop_duplicates()
    .sort_values(["R_i", "p_i"])
    .reset_index(drop=True)
)

type_pairs = list(zip(
    unique_types["R_i"].to_numpy(dtype=float),
    unique_types["p_i"].to_numpy(dtype=float),
))
type_to_idx = {pair: i for i, pair in enumerate(type_pairs)}

# 2) Observation-level vectors
s_idx_np = df_model["state_idx"].to_numpy(dtype=int)
a_obs_np = df_model["action"].to_numpy(dtype=int)
type_idx_np = np.array([
    type_to_idx[(float(R_i), float(p_i))]
    for R_i, p_i in zip(df_model["R_i"].to_numpy(), df_model["p_i"].to_numpy())
], dtype=int)

# 3) State/type vectors
h_of_state_np = np.array([s[0] for s in state_tuples], dtype=int)
a_prev_of_state_np = np.array([s[1] for s in state_tuples], dtype=int)
R_type_np = unique_types["R_i"].to_numpy(dtype=float)
p_type_np = unique_types["p_i"].to_numpy(dtype=float)

print("Fast arrays rebuilt.")
print("n_types:", len(R_type_np), "n_states:", len(h_of_state_np), "n_obs:", len(s_idx_np))

# 4) Run fast estimator
if "estimate_model_fast" not in globals():
    raise NameError("estimate_model_fast is not defined. Run the fast-function definition cell first.")

theta_init = np.array([
    0.0,    # u1_bar
    0.0,    # u2_bar
    0.0,    # alpha
    0.01,   # beta
    0.0,    # eta
    0.0, 0.0, 0.0, 0.0, 0.0  # gamma for H=5
])

results_fast = estimate_model_fast(
    theta_init=theta_init,
    H=H,
    delta=0.99,
    P=P,
    R_type=R_type_np,
    p_type=p_type_np,
    h_of_state=h_of_state_np,
    a_prev_of_state=a_prev_of_state_np,
    type_idx=type_idx_np,
    s_idx=s_idx_np,
    a_obs=a_obs_np,
    opt_method="L-BFGS-B",
)

print(results_fast["message"])
print("Converged:", results_fast["success"])
print("Iterations:", results_fast["num_iter"])
print("Neg log-likelihood:", results_fast["opt_nll"])
print(results_fast["opt_theta"])

Fast arrays rebuilt.
n_types: 8 n_states: 15 n_obs: 884
CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
Converged: True
Iterations: 161
Neg log-likelihood: 138.62811812748194
{'u1_bar': np.float64(-0.9464257234836171), 'u2_bar': np.float64(-0.3171868746645364), 'alpha': np.float64(1.5092802601125428), 'beta': np.float64(0.013517634423356302), 'eta': np.float64(1.4776582614634617), 'gamma': array([  8.86445184, -12.20750382,  -2.94104532,   1.14771365,
         4.18995794])}


In [ ]:
# Likelihood objective for NFXP-style estimation


def unpack_theta(theta_vec, H):
    """
    Parameter vector ordering:
      [u1_bar, u2_bar, alpha, beta, eta, gamma_0, ..., gamma_{H-1}]
    """
    theta_vec = np.asarray(theta_vec, dtype=float).reshape(-1)
    if theta_vec.size != 5 + H:
        raise ValueError(f"theta_vec must have length {5 + H} when H={H}.")

    return {
        "u1_bar": theta_vec[0],
        "u2_bar": theta_vec[1],
        "alpha": theta_vec[2],
        "beta": theta_vec[3],
        "eta": theta_vec[4],
        "gamma": theta_vec[5:5 + H],
    }


def build_type_ccp_map(theta, df_panel, state_tuples, P, H, delta):
    """
    Solve Bellman separately for each (R_i, p_i) type.
    Returns dict: (R_i, p_i) -> CCP array of shape (n_states, 3).
    """
    types = (
        df_panel[["R_i", "p_i"]]
        .drop_duplicates()
        .sort_values(["R_i", "p_i"])
        .itertuples(index=False)
    )

    ccp_map = {}
    for t in types:
        R_i, p_i = float(t.R_i), float(t.p_i)
        U = flow_utility_by_state(theta, state_tuples, H, R_i=R_i, p_i=p_i)
        Nu = solve_fixed_point(U, P, delta=delta)
        ccp_map[(R_i, p_i)] = choice_probabilities(Nu)
    return ccp_map


def neg_loglik(theta_vec, df_panel, state_tuples, state_to_idx, P, H, delta):
    """
    Compute negative log-likelihood:
      1) solve Bellman fixed point by (R_i, p_i) type
      2) evaluate observed action probabilities in the sample
    """
    theta = unpack_theta(theta_vec, H)
    ccp_map = build_type_ccp_map(theta, df_panel, state_tuples, P, H, delta)

    s_idx = df_panel["state_idx"].to_numpy(dtype=int)
    a_obs = df_panel["action"].to_numpy(dtype=int)
    types = list(zip(df_panel["R_i"].astype(float), df_panel["p_i"].astype(float)))

    p_obs = np.empty(len(df_panel), dtype=float)
    for i, ((R_i, p_i), s, a) in enumerate(zip(types, s_idx, a_obs)):
        p_obs[i] = ccp_map[(R_i, p_i)][s, a]

    # Numerical guard against log(0)
    p_obs = np.clip(p_obs, 1e-300, 1.0)

    return -np.sum(np.log(p_obs))


def estimate_model(
    df_panel,
    state_tuples,
    state_to_idx,
    P,
    H,
    delta=0.99,
    theta_init=None,
    opt_method="L-BFGS-B",
):
    """
    Wrapper for structural estimation with scipy.optimize.minimize.
    """
    if theta_init is None:
        theta_init = np.zeros(5 + H)

    result = minimize(
        fun=neg_loglik,
        x0=np.asarray(theta_init, dtype=float),
        args=(df_panel, state_tuples, state_to_idx, P, H, delta),
        method=opt_method,
    )

    output = {
        "success": result.success,
        "message": result.message,
        "opt_nll": float(result.fun),
        "opt_theta_vec": result.x,
        "opt_theta": unpack_theta(result.x, H),
        "num_iter": result.nit,
    }
    return output

In [ ]:
theta_init = np.array([
    0.0,    # u1_bar
    0.0,    # u2_bar
    0.0,    # alpha
    0.01,   # beta (price enters as -beta * p)
    0.0,    # eta
    0.0, 0.0, 0.0, 0.0, 0.0  # gamma for H=5 bins
])

results = estimate_model(
    df_panel=df_model,
    state_tuples=state_tuples,
    state_to_idx=state_to_idx,
    P=P,
    H=H,
    delta=0.99,
    theta_init=theta_init,
    opt_method="L-BFGS-B",
)

print(results["message"])
print("Converged:", results["success"])
print("Iterations:", results["num_iter"])
print("Neg log-likelihood:", results["opt_nll"])
print(results["opt_theta"])